# Silver Layer Parquet Analysis Notebook

This notebook reads and analyzes Parquet files from the Silver layer of the data lake.
- **Source Data**: Reddit posts and La Silla Vacía articles (Bronze layer)
- **Processing**: Pandas preprocessing with data cleaning and validation (Silver layer)
- **Output**: Parquet files with traceability metadata

## Objectives:
1. Load Parquet files using Pandas
2. Perform exploratory data analysis (EDA)
3. Validate data quality
4. Verify traceability to source files
5. Generate summary statistics

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

## 1. Define File Paths

Set up paths to locate Parquet files in the Silver layer

In [ ]:
# Define base paths
WORKSHOP_PATH = Path(".")
SILVER_PATH = WORKSHOP_PATH / "datalake_silver"
BRONZE_PATH = WORKSHOP_PATH / "datalake_bronze"

# Display paths
print("📁 Data Lake Paths:")
print(f"  Silver (Input):  {SILVER_PATH.resolve()}")
print(f"  Bronze (Source): {BRONZE_PATH.resolve()}")

# List available Parquet files
if SILVER_PATH.exists():
    parquet_files = sorted(SILVER_PATH.glob("*.parquet"))
    print(f"\n📊 Found {len(parquet_files)} Parquet files in Silver layer:")
    for pf in parquet_files:
        size_mb = pf.stat().st_size / (1024**2)
        print(f"  • {pf.name:50} ({size_mb:8.2f} MB)")
else:
    print(f"⚠️ Silver path does not exist: {SILVER_PATH.resolve()}")

## 2. Read Parquet Files

Load Parquet files using Pandas and combine them for analysis

In [ ]:
# Read Reddit Parquet files
print("📖 Reading Parquet files...\n")

reddit_files = sorted(SILVER_PATH.glob("reddit_*.parquet"))
lsv_files = sorted(SILVER_PATH.glob("lasillavacia_*.parquet"))

dfs_reddit = []
dfs_lsv = []
metadata = {}

# Load Reddit data
if reddit_files:
    print(f"🔴 Reddit Parquet files: {len(reddit_files)}")
    for pf in reddit_files:
        try:
            df = pd.read_parquet(pf)
            dfs_reddit.append(df)
            print(f"  ✅ {pf.name}: {len(df)} rows × {len(df.columns)} columns")
            if df.attrs:
                metadata[pf.name] = df.attrs
        except Exception as e:
            print(f"  ❌ {pf.name}: Error - {str(e)}")
    
    # Combine all Reddit data
    if dfs_reddit:
        df_reddit = pd.concat(dfs_reddit, ignore_index=True)
        print(f"\n  📊 Combined Reddit: {len(df_reddit)} rows × {len(df_reddit.columns)} columns\n")
else:
    print("  ⚠️ No Reddit Parquet files found\n")
    df_reddit = None

# Load La Silla Vacía data
if lsv_files:
    print(f"📰 La Silla Vacía Parquet files: {len(lsv_files)}")
    for pf in lsv_files:
        try:
            df = pd.read_parquet(pf)
            dfs_lsv.append(df)
            print(f"  ✅ {pf.name}: {len(df)} rows × {len(df.columns)} columns")
            if df.attrs:
                metadata[pf.name] = df.attrs
        except Exception as e:
            print(f"  ❌ {pf.name}: Error - {str(e)}")
    
    # Combine all La Silla Vacía data
    if dfs_lsv:
        df_lsv = pd.concat(dfs_lsv, ignore_index=True)
        print(f"\n  📊 Combined La Silla Vacía: {len(df_lsv)} rows × {len(df_lsv.columns)} columns\n")
else:
    print("  ⚠️ No La Silla Vacía Parquet files found\n")
    df_lsv = None

print("✅ All Parquet files loaded successfully")

## 3. Display Data Information

Examine the structure, data types, and metadata of loaded DataFrames

In [ ]:
# Reddit Data Information
if df_reddit is not None:
    print("=" * 80)
    print("🔴 REDDIT DATA STRUCTURE")
    print("=" * 80)
    print(f"\nShape: {df_reddit.shape}")
    print(f"Memory usage: {df_reddit.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    print("\n📋 Column Information:")
    print(df_reddit.info())
    
    print("\n📊 Data Types:")
    print(df_reddit.dtypes)
    
    print("\n📈 Statistical Summary:")
    print(df_reddit.describe(include='all').T)

# La Silla Vacía Data Information
if df_lsv is not None:
    print("\n" + "=" * 80)
    print("📰 LA SILLA VACÍA DATA STRUCTURE")
    print("=" * 80)
    print(f"\nShape: {df_lsv.shape}")
    print(f"Memory usage: {df_lsv.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    print("\n📋 Column Information:")
    print(df_lsv.info())
    
    print("\n📊 Data Types:")
    print(df_lsv.dtypes)
    
    print("\n📈 Statistical Summary:")
    print(df_lsv.describe(include='all').T)

## 4. Preview Data

Display first and last rows of each dataset

In [ ]:
# Reddit First Rows
if df_reddit is not None:
    print("=" * 80)
    print("🔴 REDDIT - First 5 Rows")
    print("=" * 80)
    display(df_reddit.head())
    
    print("\n🔴 REDDIT - Last 5 Rows")
    print("=" * 80)
    display(df_reddit.tail())

# La Silla Vacía First Rows
if df_lsv is not None:
    print("\n" + "=" * 80)
    print("📰 LA SILLA VACÍA - First 5 Rows")
    print("=" * 80)
    display(df_lsv.head())
    
    print("\n📰 LA SILLA VACÍA - Last 5 Rows")
    print("=" * 80)
    display(df_lsv.tail())

## 5. Verify Traceability

Check traceability columns (source_file, source_path) to link Silver data back to Bronze origin

In [ ]:
# Verify Traceability - Reddit
if df_reddit is not None and 'source_file' in df_reddit.columns:
    print("=" * 80)
    print("🔴 REDDIT - TRACEABILITY VERIFICATION")
    print("=" * 80)
    
    if 'source_file' in df_reddit.columns:
        print(f"\n✅ source_file column found")
        print(f"   Unique source files: {df_reddit['source_file'].nunique()}")
        print(f"\n   Distribution of records by source file:")
        print(df_reddit['source_file'].value_counts())
    
    if 'source_path' in df_reddit.columns:
        print(f"\n✅ source_path column found")
        print(f"   Sample source paths:")
        for idx, path in enumerate(df_reddit['source_path'].unique()[:3]):
            print(f"     {idx + 1}. {path}")
else:
    print("⚠️ Traceability columns not found in Reddit data")

# Verify Traceability - La Silla Vacía
if df_lsv is not None and 'source_file' in df_lsv.columns:
    print("\n" + "=" * 80)
    print("📰 LA SILLA VACÍA - TRACEABILITY VERIFICATION")
    print("=" * 80)
    
    if 'source_file' in df_lsv.columns:
        print(f"\n✅ source_file column found")
        print(f"   Unique source files: {df_lsv['source_file'].nunique()}")
        print(f"\n   Distribution of records by source file:")
        print(df_lsv['source_file'].value_counts())
    
    if 'source_path' in df_lsv.columns:
        print(f"\n✅ source_path column found")
        print(f"   Sample source paths:")
        for idx, path in enumerate(df_lsv['source_path'].unique()[:3]):
            print(f"     {idx + 1}. {path}")
else:
    print("⚠️ Traceability columns not found in La Silla Vacía data")

## 6. Data Quality Analysis

Analyze null values, duplicates, and data distribution

In [ ]:
# Data Quality Analysis - Reddit
if df_reddit is not None:
    print("=" * 80)
    print("🔴 REDDIT - DATA QUALITY ANALYSIS")
    print("=" * 80)
    
    print("\n📊 Null Values:")
    null_counts = df_reddit.isnull().sum()
    null_pct = (null_counts / len(df_reddit) * 100).round(2)
    for col, count in null_counts.items():
        if count > 0:
            print(f"  • {col}: {count} ({null_pct[col]}%)")
    
    if null_counts.sum() == 0:
        print("  ✅ No null values found!")
    
    print(f"\n🔍 Duplicate Records:")
    duplicates = df_reddit.duplicated().sum()
    print(f"  • Total duplicates: {duplicates}")
    if duplicates > 0:
        print(f"    ({duplicates / len(df_reddit) * 100:.2f}% of data)")
    else:
        print(f"  ✅ No duplicate records found!")
    
    print(f"\n📈 Key Statistics:")
    if 'score' in df_reddit.columns:
        print(f"  • Score - Mean: {df_reddit['score'].mean():.2f}, Median: {df_reddit['score'].median():.2f}")
    if 'date' in df_reddit.columns:
        print(f"  • Date range: {df_reddit['date'].min()} to {df_reddit['date'].max()}")
    if 'author' in df_reddit.columns:
        print(f"  • Unique authors: {df_reddit['author'].nunique()}")

# Data Quality Analysis - La Silla Vacía
if df_lsv is not None:
    print("\n" + "=" * 80)
    print("📰 LA SILLA VACÍA - DATA QUALITY ANALYSIS")
    print("=" * 80)
    
    print("\n📊 Null Values:")
    null_counts = df_lsv.isnull().sum()
    null_pct = (null_counts / len(df_lsv) * 100).round(2)
    for col, count in null_counts.items():
        if count > 0:
            print(f"  • {col}: {count} ({null_pct[col]}%)")
    
    if null_counts.sum() == 0:
        print("  ✅ No null values found!")
    
    print(f"\n🔍 Duplicate Records:")
    duplicates = df_lsv.duplicated().sum()
    print(f"  • Total duplicates: {duplicates}")
    if duplicates > 0:
        print(f"    ({duplicates / len(df_lsv) * 100:.2f}% of data)")
    else:
        print(f"  ✅ No duplicate records found!")
    
    print(f"\n📈 Key Statistics:")
    if 'fecha' in df_lsv.columns:
        print(f"  • Date range: {df_lsv['fecha'].min()} to {df_lsv['fecha'].max()}")
    if 'autor' in df_lsv.columns:
        print(f"  • Unique authors: {df_lsv['autor'].nunique()}")
    if 'contenido_length' in df_lsv.columns:
        print(f"  • Content length - Mean: {df_lsv['contenido_length'].mean():.0f}, Max: {df_lsv['contenido_length'].max()}")

## 7. Summary and Conclusions

Final summary of the data analysis

In [ ]:
print("=" * 80)
print("📋 ANALYSIS SUMMARY")
print("=" * 80)

print("\n✅ COMPLETION STATUS:")

# Reddit Summary
if df_reddit is not None:
    print(f"\n🔴 REDDIT DATA:")
    print(f"   • Total records: {len(df_reddit):,}")
    print(f"   • Total columns: {len(df_reddit.columns)}")
    print(f"   • Memory usage: {df_reddit.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"   • Date range: {df_reddit['date'].min() if 'date' in df_reddit.columns else 'N/A'}")
    print(f"                 {df_reddit['date'].max() if 'date' in df_reddit.columns else 'N/A'}")
    print(f"   • Data quality: ✅ Ready for analysis")
else:
    print(f"\n🔴 REDDIT DATA: ⚠️ Not available")

# La Silla Vacía Summary
if df_lsv is not None:
    print(f"\n📰 LA SILLA VACÍA DATA:")
    print(f"   • Total records: {len(df_lsv):,}")
    print(f"   • Total columns: {len(df_lsv.columns)}")
    print(f"   • Memory usage: {df_lsv.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"   • Date range: {df_lsv['fecha'].min() if 'fecha' in df_lsv.columns else 'N/A'}")
    print(f"                 {df_lsv['fecha'].max() if 'fecha' in df_lsv.columns else 'N/A'}")
    print(f"   • Data quality: ✅ Ready for analysis")
else:
    print(f"\n📰 LA SILLA VACÍA DATA: ⚠️ Not available")

print("\n" + "=" * 80)
print("🎉 NOTEBOOK ANALYSIS COMPLETE!")
print("=" * 80)
print("\nNext steps:")
print("  1. Export cleaned data for further analysis")
print("  2. Create visualizations using matplotlib or plotly")
print("  3. Perform sentiment analysis on text columns")
print("  4. Generate reports for stakeholders")